<a href="https://colab.research.google.com/github/Ger-truda/linear-regressoin-model/blob/main/LinearRegressionModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Весь алгоритм я писал самостоятельно, опираясь на математические определения и формулы, фигурирующие в алгоритме линейной регрессии.*


> Нейросети были задействованы только лишь при форматировании кода и расстановке комментариев!



In [1]:
import random
import enum

In [2]:
# Исходные данные: [Количество комнат; Площадь дома; Стоимость]
input_data = [
    [1, 30, 155],
    [2, 50, 197],
    [3, 71, 244],
    [5, 119, 356],
    [6, 140, 407],
    [7, 159, 448],
    [8, 169, 500]
]

In [3]:
# Извлечение меток (целевой переменной - стоимость) из исходных данных
labels = list(map(lambda row: row[-1], input_data))

# Извлечение признаков (количество комнат, площадь) из исходных данных
features = list(map(lambda row: row[:-1], input_data))

In [4]:
def minmax_scale(features_list):
    # Получаем размерность вектора признаков на основе первого объекта
    example_features = features_list[0]
    features_size = len(example_features)

    # Транспонирование матрицы признаков для удобной обработки по колонкам
    features_list_transpon = []
    minmax_matrix = []

    for i in range(0, features_size):
        features_list_transpon.append([])
        for obj in features_list:
            features_list_transpon[i].append(obj[i])

        # Вычисление минимального и максимального значения для текущего признака
        feature_min = min(features_list_transpon[i])
        feature_max = max(features_list_transpon[i])
        minmax_matrix.append([feature_min, feature_max])

        # Нормализация значений признака по формуле (x - min) / (max - min)
        for j, feature in enumerate(features_list_transpon[i]):
            features_list_transpon[i][j] = (feature - feature_min) / (feature_max - feature_min)

    # Обратное транспонирование для восстановления исходной структуры объектов
    result = []
    for i in range(0, len(features_list)):
        result.append([])
        for j in range(0, features_size):
            result[i].append(features_list_transpon[j][i])

    return result, minmax_matrix

In [5]:
def linear_regression(features_list, labels_list, learning_rate=0.002, epochs=1000):
    # Инициализация размерности признаков
    example_features = features_list[0]
    features_size = len(example_features)

    # Случайная инициализация весов и смещения (bias)
    weights_list = []
    offset = random.random()
    for i in range(0, features_size):
        weights_list.append(random.random())

    # Цикл обучения (стохастический градиентный спуск)
    for epoch in range(1, epochs):
        # Выбор случайного объекта из выборки для обновления весов
        random_index = random.randint(0, len(features_list) - 1)
        target_object = features_list[random_index]
        target_label = labels_list[random_index]

        # Вычисление предсказания (скалярное произведение весов и признаков + смещение)
        features_sum = 0
        for i, feature in enumerate(target_object):
            features_sum = features_sum + weights_list[i] * feature

        predicated_label = features_sum + offset

        # Вывод информации, если ошибка предсказания мала (по модулю меньше 5)
        if (abs(target_label - predicated_label) < 5):
            print(epoch, target_label - predicated_label)

        # Обновление смещения
        offset += learning_rate * (target_label - predicated_label)

        # Обновление весов
        for i, feature in enumerate(target_object):
            weights_list[i] += learning_rate * feature * (target_label - predicated_label)

    return weights_list, offset

In [6]:
def linear_regression_with_grad(features_list, labels_list, batch_size=2, learning_rate=0.002, epochs=1000):
    # Инициализация размерности признаков
    example_features = features_list[0]
    features_size = len(example_features)

    # Случайная инициализация весов и смещения
    weights_list = []
    offset = random.random()

    # Разделение данных на батчи (пакеты)
    batches_num = round(len(labels_list) / batch_size)
    batches = []
    for num in range(0, batches_num):
        start_idx = num * batch_size
        end_idx = start_idx + batch_size
        batches.append([features_list[start_idx:end_idx], labels_list[start_idx:end_idx]])

    for i in range(0, features_size):
        weights_list.append(random.random())

    # Цикл обучения (мини-пакетный градиентный спуск)
    for epoch in range(1, epochs):
        # Случайный выбор батча на каждой эпохе
        random_index = random.randint(0, len(batches) - 1)
        se = 0
        target_objects = batches[random_index][0]
        target_labels = batches[random_index][1]

        # Накопление ошибки и градиента по батчу
        for i, target_object in enumerate(target_objects):
            target_label = target_labels[i]
            features_sum = 0
            for i, feature in enumerate(target_object):
                features_sum = features_sum + weights_list[i] * feature

            predicated_label = features_sum + offset
            # Вычисление производной ошибки (суммируется по батчу)
            se += 2 * (target_label - predicated_label) * sum(target_object)

        # Усреднение градиента по размеру батча (MSE)
        mse = se / len(batches[random_index][0])

        # Обновление смещения на основе среднего градиента
        offset += learning_rate * mse

        # Обновление весов на основе среднего градиента
        for i, feature in enumerate(target_object):
            weights_list[i] += learning_rate * feature * mse

    return weights_list, offset

In [8]:
class LinearRegressionModel:
    def __init__(self):
        # Инициализация параметров модели
        self.weights_list = []  # Веса признаков
        self.offset = 0         # Смещение (bias)
        self.minmax_matrix = [] # Матрица мин/макс значений для нормализации

    def lern(self, features_list, labels_list, batch_size=1, learning_rate=0.002, epochs=10000):
        # Нормализация входных признаков
        prepared_features, minmax_matrix = minmax_scale(features_list)
        self.minmax_matrix = minmax_matrix

        # Обучение модели
        weights_list, offset = linear_regression_with_grad(prepared_features, labels_list, batch_size, learning_rate, epochs)

        # Сохранение обученных параметров
        self.weights_list = weights_list
        self.offset = offset

    def predicate(self, features_list):
        # Вычисление предсказания для новых данных
        features_sum = 0
        for i, feature in enumerate(features_list):
            # Нормализация входного признака с использованием сохраненных min/max значений
            normalize_feature = (feature - self.minmax_matrix[i][0]) / (self.minmax_matrix[i][1] - self.minmax_matrix[i][0])
            features_sum = features_sum + self.weights_list[i] * normalize_feature

        # Итоговое предсказание
        predicated_label = features_sum + self.offset
        return predicated_label

In [9]:
linear_regression_model = LinearRegressionModel()
linear_regression_model.lern(features, labels)

In [10]:
predicated_price = linear_regression_model.predicate([4, 92])
print(predicated_price)

297.89814942924426
